# OCR BNP — Pipeline V11 — Batch GPU + JSON checkpoint + Stats
> `Kernel → Restart Kernel and Run All Cells`

**Améliorations V11 :**
- Batch GPU : plusieurs pages traitées en parallèle (x3-4 plus rapide)
- JSON sauvegardé après chaque dossier → reprise automatique si crash
- Tokens consommés + temps enregistrés dans chaque JSON
- Numéro séquentiel dans le log


## 1. Dépendances

In [ ]:
%pip install -q pymupdf pillow transformers openpyxl
print('✅ OK')

## 2. Imports

In [ ]:
import time, json, re
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import fitz
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForVision2Seq
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
print('✅ Imports OK')

## 3. Config

In [ ]:
MODEL_PATH      = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen2.5-VL-7B-Instruct/main"
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS  = 700
IMAGE_MAX_SIZE  = 1120
PDF_ZOOM        = 2.0
BLANK_THRESHOLD = 0.97
GPU_BATCH_SIZE  = 4      # Nombre de pages traitées en parallèle — ajuster selon VRAM

INPUT_DIR  = Path("/mnt/data/transferts_in")
OUTPUT_DIR = Path("/mnt/data/transferts_out")
JSON_DIR   = OUTPUT_DIR / "json_dossiers"
LOG_PATH   = OUTPUT_DIR / "pipeline.log"
EXCEL_PATH = OUTPUT_DIR / f"audit_transferts_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)

pdfs = sorted(INPUT_DIR.glob("*.pdf"))
print(f'Device         : {DEVICE}')
print(f'GPU batch size : {GPU_BATCH_SIZE}')
print(f'Dossiers       : {len(pdfs)}')
print(f'JSON dir       : {JSON_DIR}')
print(f'Excel          : {EXCEL_PATH}')

## 4. Chargement modèle

In [ ]:
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16,
    trust_remote_code=True, low_cpu_mem_usage=True,
)
model.eval().to(DEVICE)
print(f'✅ Modèle chargé en {time.time()-t0:.1f}s')

## 5. Utilitaires PDF & inférence

In [ ]:
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
import gc

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)


def build_transform(input_size=448):
    return T.Compose([
        T.Lambda(lambda img: img.convert("RGB")),
        T.Resize((input_size, input_size),
                  interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def resize(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side: return img
    r = max_side / max(w, h)
    return img.resize((int(w*r), int(h*r)), Image.LANCZOS)


def is_blank(image, threshold=BLANK_THRESHOLD) -> bool:
    arr = np.array(image.convert("L"))
    return (arr > 240).sum() / arr.size >= threshold


def pdf_to_pages(path: Path, zoom=PDF_ZOOM) -> list:
    doc    = fitz.open(path)
    matrix = fitz.Matrix(zoom, zoom)
    pages  = []
    for i in range(len(doc)):
        pix = doc.load_page(i).get_pixmap(matrix=matrix, alpha=False)
        img = resize(Image.frombytes("RGB", [pix.width, pix.height], pix.samples))
        pages.append({"index": i, "image": img})
    doc.close()
    return pages


def parse_json(text: str) -> dict:
    if not text:
        return {}
    text = text.strip().replace("", "").strip()
    try:
        m = re.search(r"\{.*\}", text, re.S)
        return json.loads(m.group()) if m else {}
    except Exception:
        return {}


def _encode_image(image: "Image.Image") -> tuple:
    """Encode une image PIL pour InternVL3."""
    transform    = build_transform(input_size=448)
    pixel_values = transform(image).unsqueeze(0).to(torch.float16).to(DEVICE)
    return pixel_values


def ask_single(prompt: str, image: "Image.Image") -> dict:
    """
    Inférence InternVL3 — 1 image.
    Retourne {text, tokens_in, tokens_out, elapsed}.
    """
    pixel_values = _encode_image(image)
    question     = f"<image>
{prompt}"
    enc          = tokenizer(question, return_tensors="pt",
                              truncation=True, max_length=4096).to(DEVICE)

    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            pixel_values   = pixel_values,
            input_ids      = enc.input_ids,
            attention_mask = enc.attention_mask,
            max_new_tokens = MAX_NEW_TOKENS,
            do_sample      = False,
            eos_token_id   = tokenizer.eos_token_id,
            pad_token_id   = tokenizer.eos_token_id,
        )
    elapsed  = time.time() - t0
    in_len   = enc.input_ids.shape[1]
    generated = outputs[0][in_len:]
    text      = tokenizer.decode(generated, skip_special_tokens=True,
                                  clean_up_tokenization_spaces=True)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "text":       text,
        "tokens_in":  int(in_len),
        "tokens_out": int(len(generated)),
        "elapsed":    round(elapsed, 2),
    }


def ask_batch(prompt: str, images: list) -> list:
    """
    Batch InternVL3 — N images en parallèle.
    Fallback ask_single si OOM.
    """
    if not images:
        return []
    if len(images) == 1:
        return [ask_single(prompt, images[0])]

    try:
        transform    = build_transform(input_size=448)
        pixel_values = torch.stack([
            transform(img).to(torch.float16) for img in images
        ]).to(DEVICE)

        questions = [f"<image>
{prompt}"] * len(images)
        enc       = tokenizer(
            questions, return_tensors="pt",
            padding=True, truncation=True, max_length=4096
        ).to(DEVICE)

        t0 = time.time()
        with torch.no_grad():
            outputs = model.generate(
                pixel_values   = pixel_values,
                input_ids      = enc.input_ids,
                attention_mask = enc.attention_mask,
                max_new_tokens = MAX_NEW_TOKENS,
                do_sample      = False,
                eos_token_id   = tokenizer.eos_token_id,
                pad_token_id   = tokenizer.eos_token_id,
            )
        elapsed_total = time.time() - t0
        in_len        = enc.input_ids.shape[1]

        results = []
        for i in range(len(images)):
            generated = outputs[i][in_len:]
            text = tokenizer.decode(generated, skip_special_tokens=True,
                                     clean_up_tokenization_spaces=True)
            results.append({
                "text":       text,
                "tokens_in":  int(in_len),
                "tokens_out": int(len(generated)),
                "elapsed":    round(elapsed_total / len(images), 2),
            })

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return results

    except torch.cuda.OutOfMemoryError:
        print(f"  ⚠️  OOM batch → fallback page par page")
        gc.collect()
        torch.cuda.empty_cache()
        return [ask_single(prompt, img) for img in images]


print(f"✅ Utilitaires InternVL3 OK")
print(f"   Device     : {DEVICE}")
print(f"   Max tokens : {MAX_NEW_TOKENS}")


## 6. Normalisation des données

In [ ]:
def norm_str(v) -> str:
    if v is None: return None
    s = re.sub(r'\s+', ' ', str(v).strip())
    return s if s and s.lower() not in ('null', 'none', 'n/a') else None

def norm_upper(v) -> str:
    s = norm_str(v)
    return s.upper() if s else None

def norm_compte(v) -> str:
    s = norm_str(v)
    if not s: return None
    return re.sub(r'[^A-Za-z0-9]', '', s).upper()

def norm_montant(v) -> float:
    if v is None: return None
    if isinstance(v, (int, float)): return float(v)
    s = str(v).strip()
    s = re.sub(r'[^\d.,]', '', s)
    if not s: return None
    if s.count(',') == 1 and '.' not in s: s = s.replace(',', '.')
    elif '.' in s and ',' in s:            s = s.replace(',', '')
    elif s.count(',') > 1:                 s = s.replace(',', '')
    try: return float(s)
    except: return None

def norm_date(v) -> str:
    if not v: return None
    s = norm_str(v)
    if not s: return None
    if re.match(r'^\d{2}/\d{2}/\d{4}$', s): return s
    m = re.match(r'^(\d{4})-(\d{2})-(\d{2})$', s)
    if m: return f'{m.group(3)}/{m.group(2)}/{m.group(1)}'
    return s

def norm_periode(v) -> str:
    if not v: return None
    s = str(v).upper().strip()
    s = re.sub(r'[._\-]', ' ', s)
    s = re.sub(r'PART\s*(\d)', r'PART \1', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


# Whitelists — champs autorisés par type (anti-parasite)
CHAMPS_OV = {
    'type', 'monnaie', 'montant_chiffres', 'montant_lettres',
    'periode', 'tranche', 'date_demande', 'compte_debiteur',
    'beneficiaire_nom', 'beneficiaire_compte', 'beneficiaire_adresse',
    'swift', 'banque_nom'
}
CHAMPS_ANN1 = {
    'type', 'nom_prenom_client', 'compte_bancaire',
    'nom_prenom_signataire', 'date_signature'
}
CHAMPS_ANN2 = {
    'type', 'mois_transfert', 'nom_prenom', 'compte_bancaire',
    'salaire_mensuel', 'nombre_jours', 'part_transferable',
    'pays_destination', 'nom_banque', 'numero_compte_etranger',
    'numero_domiciliation'
}
CHAMPS_BP = {
    'type', 'nom_prenom_salarie', 'matricule', 'mois_bulletin',
    'salaire_base', 'salaire_brut', 'retenue_ss', 'retenue_irg',
    'retenue_mutuelle', 'net_a_payer'
}


def normalise_doc(doc_type: str, data: dict) -> dict:
    d = dict(data)

    if doc_type == 'OV':
        # Aplatissement zones imbriquées
        for key in list(d.keys()):
            if isinstance(d.get(key), dict):
                d.update(d.pop(key))
        # Fallback nom alternatif compte_debiteur
        if not d.get('compte_debiteur'):
            for alias in ['compte_debiteur_donneur_ordre', 'compte_donneur',
                          'numero_compte', 'compte_debiteur_ordre']:
                if d.get(alias):
                    d['compte_debiteur'] = d[alias]
                    break
        # Corriger tranche = numéro de zone
        if d.get('tranche') in ('70','32','50','59','57', 70, 32, 50, 59, 57):
            d['tranche'] = None
        # Corriger periode = date (format dd/mm/yyyy)
        if d.get('periode') and re.match(r'^\d{2}/\d{2}/\d{4}$', str(d.get('periode',''))):
            d['periode'] = None
        # Whitelist
        d = {k: v for k, v in d.items() if k in CHAMPS_OV}
        # Normalisation
        d['monnaie']              = norm_upper(d.get('monnaie'))
        d['montant_chiffres']     = norm_montant(d.get('montant_chiffres'))
        d['montant_lettres']      = norm_str(d.get('montant_lettres'))
        d['periode']              = norm_periode(d.get('periode'))
        d['tranche']              = norm_str(d.get('tranche'))
        d['date_demande']         = norm_date(d.get('date_demande'))
        d['compte_debiteur']      = norm_compte(d.get('compte_debiteur'))
        d['beneficiaire_nom']     = norm_upper(d.get('beneficiaire_nom'))
        d['beneficiaire_compte']  = norm_compte(d.get('beneficiaire_compte'))
        d['beneficiaire_adresse'] = norm_str(d.get('beneficiaire_adresse'))
        d['swift']                = norm_compte(d.get('swift'))
        d['banque_nom']           = norm_upper(d.get('banque_nom'))

    elif doc_type == 'ANNEXE_I':
        d = {k: v for k, v in d.items() if k in CHAMPS_ANN1}
        d['nom_prenom_client']    = norm_upper(d.get('nom_prenom_client'))
        d['compte_bancaire']      = norm_compte(d.get('compte_bancaire'))
        d['nom_prenom_signataire']= norm_upper(d.get('nom_prenom_signataire'))
        d['date_signature']       = norm_date(d.get('date_signature'))

    elif doc_type == 'ANNEXE_II':
        d = {k: v for k, v in d.items() if k in CHAMPS_ANN2}
        d['mois_transfert']         = norm_periode(d.get('mois_transfert'))
        d['nom_prenom']             = norm_upper(d.get('nom_prenom'))
        d['compte_bancaire']        = norm_compte(d.get('compte_bancaire'))
        d['salaire_mensuel']        = norm_montant(d.get('salaire_mensuel'))
        d['nombre_jours']           = norm_str(d.get('nombre_jours'))
        d['part_transferable']      = norm_montant(d.get('part_transferable'))
        d['pays_destination']       = norm_upper(d.get('pays_destination'))
        d['nom_banque']             = norm_str(d.get('nom_banque'))
        d['numero_compte_etranger'] = norm_compte(d.get('numero_compte_etranger'))
        d['numero_domiciliation']   = norm_str(d.get('numero_domiciliation'))

    elif doc_type == 'BULLETIN':
        d = {k: v for k, v in d.items() if k in CHAMPS_BP}
        d['nom_prenom_salarie'] = norm_upper(d.get('nom_prenom_salarie'))
        d['matricule']          = norm_str(d.get('matricule'))
        d['mois_bulletin']      = norm_periode(d.get('mois_bulletin'))
        d['salaire_base']       = norm_montant(d.get('salaire_base'))
        d['salaire_brut']       = norm_montant(d.get('salaire_brut'))
        d['retenue_ss']         = norm_montant(d.get('retenue_ss'))
        d['retenue_irg']        = norm_montant(d.get('retenue_irg'))
        d['retenue_mutuelle']   = norm_montant(d.get('retenue_mutuelle'))
        d['net_a_payer']        = norm_montant(d.get('net_a_payer'))

    return d


print('✅ Normalisation OK')

## 7. Prompt universel

In [ ]:
PROMPT_UNIVERSEL = """\
Lis ce document bancaire.

ÉTAPE 1 — Identifie le type en lisant le titre principal :
- OV        : titre "ORDRE DE VIREMENT A L'ETRANGER"
- ANNEXE_I  : titre "Annexe I", contient "Transfert sur salaire perçus en Algérie"
- ANNEXE_II : titre "Annexe II", contient "Fiche de paie spéciale relative à un transfert du salaire"
- BULLETIN  : titre "BULLETIN DE PAIE"
- AUTRE     : tout autre document (page vide, tableau, cachet...)

ÉTAPE 2 — Selon le type identifié, extrais uniquement les champs listés.
Si le type est AUTRE, retourne uniquement {"type": "AUTRE"}.

--- Si OV ---
Zone 32 : monnaie, montant_chiffres, montant_lettres
Zone 70 : periode (mois + année bruts tels qu'écrits ex: JANVIER 2026 / null),
          tranche (Part 1 / Part 2 / null. Ne pas confondre avec le numéro 70 imprimé à gauche)
Zone 50 : date_demande, compte_debiteur (20 chiffres commençant par 02700)
Zone 59 : beneficiaire_nom, beneficiaire_compte (IBAN sans espaces), beneficiaire_adresse
Zone 57 : swift, banque_nom

--- Si ANNEXE_I ---
nom_prenom_client (juste après "Je soussigné"),
compte_bancaire (numéro 20 chiffres),
nom_prenom_signataire,
date_signature (après "Bethioua le", format jj/MM/aaaa)

--- Si ANNEXE_II ---
mois_transfert (valeur brute après "Mois de"),
nom_prenom,
compte_bancaire (numéro 20 chiffres),
salaire_mensuel,
nombre_jours (null si absent),
part_transferable (montant chiffres uniquement),
pays_destination,
nom_banque (première partie après "Compte devise :"),
numero_compte_etranger (deuxième partie après "Compte devise :", sans espaces),
numero_domiciliation (premier nombre du tableau "DOMICILIATION IMPORT" ex: 271901, null si absent)

--- Si BULLETIN ---
nom_prenom_salarie, matricule, mois_bulletin,
salaire_base, salaire_brut, retenue_ss, retenue_irg, retenue_mutuelle, net_a_payer

RÈGLES :
- Retourne UNIQUEMENT un JSON valide, sans texte avant ni après
- Commence toujours par {"type": "..."}
- Si un champ est absent ou illisible : null
- Pas de markdown, pas de backticks
- N'invente aucun champ supplémentaire
"""

print(f'✅ Prompt : {len(PROMPT_UNIVERSEL)} caractères')

## 8. Traitement d'un PDF (batch GPU)

In [ ]:
def process_pdf(pdf_path: Path, verbose=True) -> dict:
    """
    Traite un PDF avec batch GPU.
    Toutes les pages non-blanches sont envoyées par groupes de GPU_BATCH_SIZE.
    Retourne un dict avec données + stats (tokens, temps).
    """
    import gc
    TYPES_ATTENDUS = {'OV', 'ANNEXE_I', 'ANNEXE_II', 'BULLETIN'}

    pages    = pdf_to_pages(pdf_path)
    results  = {}
    doublons = []
    total_tok_in  = 0
    total_tok_out = 0
    t_dossier     = time.time()

    if verbose:
        print(f'\n📁 {pdf_path.name} — {len(pages)} page(s)')

    # Filtrer pages non-blanches
    pages_actives = [p for p in pages if not is_blank(p['image'])]
    pages_vides   = len(pages) - len(pages_actives)

    if verbose and pages_vides:
        print(f'  {pages_vides} page(s) vide(s) ignorée(s)')

    # Traitement par batch
    for batch_start in range(0, len(pages_actives), GPU_BATCH_SIZE):
        batch  = pages_actives[batch_start:batch_start + GPU_BATCH_SIZE]
        images = [p['image'] for p in batch]

        t_batch = time.time()
        reps    = ask_batch(PROMPT_UNIVERSEL, images)
        elapsed_batch = time.time() - t_batch

        for page, rep in zip(batch, reps):
            i        = page['index']
            data     = parse_json(rep['text'])
            doc_type = data.get('type', 'INCONNU')

            total_tok_in  += rep['tokens_in']
            total_tok_out += rep['tokens_out']

            if doc_type in ('AUTRE', 'INCONNU'):
                if verbose:
                    print(f'  Page {i+1} → {doc_type} — ignorée')
                continue

            data = normalise_doc(doc_type, data)

            if verbose:
                print(f'  Page {i+1} → {doc_type:10s} '
                      f'| tok={rep["tokens_in"]}+{rep["tokens_out"]}')
                for k, v in data.items():
                    if k != 'type' and v is not None:
                        print(f'    {k:25s}: {v}')

            if doc_type in results:
                doublons.append(doc_type)
                if verbose: print(f'    ⚠️  DOUBLON {doc_type}')
                continue

            results[doc_type] = data

        # Purge VRAM entre batches
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    pages_trouvees   = sorted(results.keys())
    pages_manquantes = sorted(TYPES_ATTENDUS - set(results.keys()))

    return {
        'fichier':          pdf_path.name,
        'date_traitement':  datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'temps_total_s':    round(time.time() - t_dossier, 2),
        'tokens_in':        total_tok_in,
        'tokens_out':       total_tok_out,
        'tokens_total':     total_tok_in + total_tok_out,
        'pages_trouvees':   ', '.join(pages_trouvees),
        'pages_manquantes': ', '.join(pages_manquantes) if pages_manquantes else None,
        'anomalies':        'DOUBLON: ' + ', '.join(doublons) if doublons else None,
        **{t: results.get(t, {}) for t in TYPES_ATTENDUS},
    }


print('✅ process_pdf OK')

## 9. Export Excel

In [ ]:
SCHEMA = {
    'META': [
        ('fichier',          'Fichier'),
        ('date_traitement',  'Date traitement'),
        ('temps_total_s',    'Temps (s)'),
        ('tokens_total',     'Tokens total'),
        ('pages_trouvees',   'Pages trouvées'),
        ('pages_manquantes', 'Pages manquantes'),
        ('anomalies',        'Anomalies'),
    ],
    'OV': [
        ('monnaie',              'Monnaie'),
        ('montant_chiffres',     'Montant (chiffres)'),
        ('montant_lettres',      'Montant (lettres)'),
        ('periode',              'Période'),
        ('tranche',              'Tranche'),
        ('date_demande',         'Date demande'),
        ('compte_debiteur',      'Compte débiteur'),
        ('beneficiaire_nom',     'Bénéficiaire nom'),
        ('beneficiaire_compte',  'Bénéficiaire compte'),
        ('beneficiaire_adresse', 'Bénéficiaire adresse'),
        ('swift',                'SWIFT'),
        ('banque_nom',           'Banque bénéficiaire'),
    ],
    'ANNEXE_I': [
        ('nom_prenom_client',     'Nom Prénom client'),
        ('compte_bancaire',       'Compte bancaire'),
        ('nom_prenom_signataire', 'Nom Prénom signataire'),
        ('date_signature',        'Date signature'),
    ],
    'ANNEXE_II': [
        ('mois_transfert',         'Mois transfert'),
        ('nom_prenom',             'Nom Prénom'),
        ('compte_bancaire',        'Compte bancaire'),
        ('salaire_mensuel',        'Salaire mensuel'),
        ('nombre_jours',           'Nombre de jours'),
        ('part_transferable',      'Part transférable'),
        ('pays_destination',       'Pays destination'),
        ('nom_banque',             'Nom banque'),
        ('numero_compte_etranger', 'Compte étranger'),
        ('numero_domiciliation',   'N° Domiciliation'),
    ],
    'BULLETIN': [
        ('nom_prenom_salarie', 'Nom Prénom'),
        ('matricule',          'Matricule'),
        ('mois_bulletin',      'Mois bulletin'),
        ('salaire_base',       'Salaire base'),
        ('salaire_brut',       'Salaire brut'),
        ('retenue_ss',         'Retenue SS'),
        ('retenue_irg',        'Retenue IRG'),
        ('retenue_mutuelle',   'Retenue mutuelle'),
        ('net_a_payer',        'Net à payer'),
    ],
}

COLORS = {
    'META':      {'header': 'FF1F4E79', 'col': 'FFD6E4F0'},
    'OV':        {'header': 'FF833C00', 'col': 'FFFCE4D6'},
    'ANNEXE_I':  {'header': 'FF375623', 'col': 'FFE2EFDA'},
    'ANNEXE_II': {'header': 'FF203864', 'col': 'FFDAE3F3'},
    'BULLETIN':  {'header': 'FF3F3151', 'col': 'FFEDE7F6'},
}


def create_excel(path: Path, rows: list):
    wb = Workbook()
    ws = wb.active
    ws.title = 'Dossiers'
    all_cols = []
    for groupe, cols in SCHEMA.items():
        for field_key, label in cols:
            all_cols.append((groupe, field_key, label))

    col_idx = 1
    group_map = defaultdict(list)
    for groupe, _, _ in all_cols:
        group_map[groupe].append(col_idx)
        col_idx += 1

    for groupe, cols in group_map.items():
        s, e = cols[0], cols[-1]
        if s < e:
            ws.merge_cells(start_row=1, start_column=s, end_row=1, end_column=e)
        c = ws.cell(row=1, column=s)
        c.value = groupe
        c.font  = Font(bold=True, color='FFFFFFFF', name='Arial', size=11)
        c.fill  = PatternFill('solid', start_color=COLORS[groupe]['header'])
        c.alignment = Alignment(horizontal='center', vertical='center')
    ws.row_dimensions[1].height = 22

    for i, (groupe, _, label) in enumerate(all_cols, start=1):
        c = ws.cell(row=2, column=i)
        c.value = label
        c.font  = Font(bold=True, name='Arial', size=9)
        c.fill  = PatternFill('solid', start_color=COLORS[groupe]['col'])
        c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        c.border = Border(bottom=Side(style='thin'), right=Side(style='hair'))
        ws.column_dimensions[get_column_letter(i)].width = 20
    ws.row_dimensions[2].height = 35
    ws.freeze_panes = ws.cell(row=3, column=len(SCHEMA['META']) + 1)

    for row_num, dossier in enumerate(rows, start=3):
        for col_idx, (groupe, field_key, _) in enumerate(all_cols, start=1):
            val = (dossier.get(field_key) if groupe == 'META'
                   else (dossier.get(groupe) or {}).get(field_key))
            c = ws.cell(row=row_num, column=col_idx)
            c.value = val
            c.font  = Font(name='Arial', size=9)
            c.fill  = PatternFill('solid', start_color=COLORS[groupe]['col'])
            if isinstance(val, float):
                c.number_format = '0.00'
            if groupe == 'META' and field_key in ('anomalies', 'pages_manquantes') and val:
                c.font = Font(name='Arial', size=9, bold=True, color='FFCC0000')

    wb.save(path)
    print(f'✅ Excel : {path} | {len(rows)} dossiers | {len(all_cols)} colonnes')


print('✅ Export Excel OK')

## 10. Log

In [ ]:
def log(msg: str):
    ligne = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} — {msg}"
    print(ligne)
    with open(LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(ligne + '\n')


print('✅ Log OK')

## 11. Pipeline complet

In [ ]:
import gc
import psutil

# ── Vérification RAM ──────────────────────────────────────────────────────────
ram_libre = psutil.virtual_memory().available / 1e9
print(f'RAM libre : {ram_libre:.1f} GB')

# ── Dossiers à traiter ────────────────────────────────────────────────────────
deja_traites = {f.stem for f in JSON_DIR.glob('*.json')}
a_traiter    = [p for p in pdfs if p.stem not in deja_traites]
total_pdfs   = len(pdfs)

log(f'Total PDFs     : {total_pdfs}')
log(f'Déjà traités   : {len(deja_traites)}')
log(f'À traiter      : {len(a_traiter)}')
log(f'GPU batch size : {GPU_BATCH_SIZE}')
log(f'RAM libre      : {ram_libre:.1f} GB')

# ── Chunk size automatique selon RAM disponible ───────────────────────────────
taille_img_mb = 1120 * 1584 * 3 / 1e6       # ~5MB par image
ram_estimee   = len(a_traiter) * 4 * taille_img_mb / 1e3
CHUNK_SIZE    = min(
    len(a_traiter),
    max(GPU_BATCH_SIZE, int(ram_libre * 0.7 * 1e3 / (4 * taille_img_mb)))
)
log(f'RAM estimée    : {ram_estimee:.1f} GB')
log(f'Chunk size     : {CHUNK_SIZE} dossiers')

chunks = [a_traiter[i:i+CHUNK_SIZE] for i in range(0, len(a_traiter), CHUNK_SIZE)]
log(f'{len(chunks)} chunk(s) de {CHUNK_SIZE} dossiers max')

# ── Pipeline ──────────────────────────────────────────────────────────────────
t_total       = time.time()
grand_tok_in  = 0
grand_tok_out = 0
n_ok = n_err  = 0
TYPES_ATTENDUS = {'OV', 'ANNEXE_I', 'ANNEXE_II', 'BULLETIN'}

for num_chunk, chunk in enumerate(chunks, start=1):
    log(f'── Chunk {num_chunk}/{len(chunks)} : {len(chunk)} dossiers ──')

    # ── Phase 1 : Chargement pages ────────────────────────────────────────────
    t_load    = time.time()
    all_pages = []
    for pdf_path in chunk:
        try:
            pages = pdf_to_pages(pdf_path)
            for page in pages:
                if not is_blank(page['image']):
                    all_pages.append((pdf_path, page))
        except Exception as e:
            log(f'❌ Chargement {pdf_path.name} : {e}')
    log(f'   {len(all_pages)} pages chargées en {time.time()-t_load:.1f}s')

    # ── Phase 2 : Inférence GPU ───────────────────────────────────────────────
    t_infer     = time.time()
    raw_results = defaultdict(list)

    for batch_start in range(0, len(all_pages), GPU_BATCH_SIZE):
        batch  = all_pages[batch_start:batch_start + GPU_BATCH_SIZE]
        images = [page['image'] for _, page in batch]
        try:
            reps = ask_batch(PROMPT_UNIVERSEL, images)
            for (pdf_path, page), rep in zip(batch, reps):
                raw_results[pdf_path].append({
                    'index':      page['index'],
                    'text':       rep['text'],
                    'tokens_in':  rep['tokens_in'],
                    'tokens_out': rep['tokens_out'],
                })
            pct = min(100, (batch_start + len(batch)) / len(all_pages) * 100)
            log(f'   {batch_start+len(batch):>5}/{len(all_pages)} pages | {pct:.0f}%')
        except Exception as e:
            log(f'❌ Batch {batch_start}-{batch_start+len(batch)} : {e}')
            continue
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    log(f'   Inférence en {time.time()-t_infer:.1f}s')

    # ── Phase 3 : Post-traitement + sauvegarde JSON ───────────────────────────
    t_post = time.time()

    for num, pdf_path in enumerate(chunk, start=1):
        try:
            pages_raw = raw_results.get(pdf_path, [])
            if not pages_raw:
                log(f'   [{num:>4}/{len(chunk)}] ⚠️  {pdf_path.name} — aucune page')
                continue

            results  = {}
            doublons = []
            total_tok_in  = 0
            total_tok_out = 0
            t_dossier = time.time()

            for page_raw in sorted(pages_raw, key=lambda x: x['index']):
                data     = parse_json(page_raw['text'])
                doc_type = data.get('type', 'INCONNU')
                total_tok_in  += page_raw['tokens_in']
                total_tok_out += page_raw['tokens_out']
                if doc_type in ('AUTRE', 'INCONNU'):
                    continue
                data = normalise_doc(doc_type, data)
                if doc_type in results:
                    doublons.append(doc_type)
                    continue
                results[doc_type] = data

            pages_trouvees   = sorted(results.keys())
            pages_manquantes = sorted(TYPES_ATTENDUS - set(results.keys()))

            result = {
                'fichier':          pdf_path.name,
                'date_traitement':  datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                'temps_total_s':    round(time.time() - t_dossier, 2),
                'tokens_in':        total_tok_in,
                'tokens_out':       total_tok_out,
                'tokens_total':     total_tok_in + total_tok_out,
                'pages_trouvees':   ', '.join(pages_trouvees),
                'pages_manquantes': ', '.join(pages_manquantes) if pages_manquantes else None,
                'anomalies':        'DOUBLON: ' + ', '.join(doublons) if doublons else None,
                **{t: results.get(t, {}) for t in TYPES_ATTENDUS},
            }

            json_file = JSON_DIR / f'{pdf_path.stem}.json'
            with open(json_file, 'w', encoding='utf-8') as f:
                json.dump(result, f, ensure_ascii=False, indent=2, default=str)

            grand_tok_in  += total_tok_in
            grand_tok_out += total_tok_out
            n_ok += 1

            msg = (f'   [{num:>4}/{len(chunk)}] ✅ {pdf_path.name}'
                   f' | tok={result["tokens_total"]}'
                   f' | {result["pages_trouvees"]}')
            if result.get('pages_manquantes'):
                msg += f' | ⚠️  {result["pages_manquantes"]}'
            if result.get('anomalies'):
                msg += f' | 🔴 {result["anomalies"]}'
            log(msg)

        except Exception as e:
            n_err += 1
            log(f'   [{num:>4}/{len(chunk)}] ❌ {pdf_path.name} — {e}')
            continue

    log(f'   Post-traitement en {time.time()-t_post:.1f}s')

    # Libérer RAM du chunk
    del all_pages, raw_results
    gc.collect()
    ram_now = psutil.virtual_memory().available / 1e9
    log(f'   RAM libre après chunk : {ram_now:.1f} GB')

# ── Reconstruction Excel ──────────────────────────────────────────────────────
log('Génération Excel...')
rows = []
for json_file in sorted(JSON_DIR.glob('*.json')):
    with open(json_file, encoding='utf-8') as f:
        rows.append(json.load(f))

create_excel(EXCEL_PATH, rows)

elapsed = time.time() - t_total
log(f'✅ Terminé en {elapsed:.1f}s ({elapsed/max(1,n_ok):.1f}s/dossier)')
log(f'   Traités      : {n_ok} | Erreurs : {n_err}')
log(f'   Tokens IN    : {grand_tok_in:,}')
log(f'   Tokens OUT   : {grand_tok_out:,}')
log(f'   Tokens TOTAL : {grand_tok_in + grand_tok_out:,}')

## 12. Récupération d'urgence

Si le kernel plante et que `rows` est encore en mémoire, exécuter cette cellule
pour sauvegarder immédiatement en JSON avant de tout perdre.

In [ ]:
# Décommenter et exécuter en cas de crash kernel
# try:
#     print(f'rows en mémoire : {len(rows)}')
#     for result in rows:
#         nom = Path(result['fichier']).stem
#         jf  = JSON_DIR / f'{nom}.json'
#         if not jf.exists():
#             with open(jf, 'w', encoding='utf-8') as f:
#                 json.dump(result, f, ensure_ascii=False, indent=2, default=str)
#     print(f'✅ Sauvegardé')
# except Exception as e:
#     print(f'rows non disponible : {e}')

## 13. Analyse des résultats

In [ ]:
import pandas as pd

if EXCEL_PATH.exists():
    df = pd.read_excel(EXCEL_PATH, header=1)
    print(f'📊 {len(df)} dossiers')

    # Stats tokens
    if 'Tokens total' in df.columns:
        print(f'   Tokens total   : {df["Tokens total"].sum():,.0f}')
        print(f'   Tokens/dossier : {df["Tokens total"].mean():,.0f}')
    if 'Temps (s)' in df.columns:
        print(f'   Temps moyen    : {df["Temps (s)"].mean():.1f}s/dossier')
    print()

    if 'Anomalies' in df.columns:
        anom = df[df['Anomalies'].notna()]
        print(f'🔴 Doublons : {len(anom)}')
        if len(anom): print(anom[['Fichier','Anomalies']].to_string(index=False))
        print()

    if 'Pages manquantes' in df.columns:
        manq = df[df['Pages manquantes'].notna()]
        print(f'⚠️  Incomplets : {len(manq)}')
        if len(manq): print(manq[['Fichier','Pages manquantes']].to_string(index=False))
        print()

    display(df.head(5))
else:
    print('Excel non encore généré.')